# 第2回 演習: LoRA で用語集の質問応答を学習する

`glossary_sft.jsonl` の会話データを使って小さな指示チューニング済みモデルを LoRA で学習し、質問の言い換えに対する学習前後の回答を比べます。評価では、生成した定義文が `gold` の正解と一致する割合を見ます。

**この演習で確かめること**

1. 学習前のモデルが用語の質問にどう答えるかを記録する（セクション 3）。
2. `messages` 形式のデータを `Dataset.from_list` から SFTTrainer に渡して LoRA 学習する（セクション 4）。
3. 学習で使わなかった質問表現で、定義文を再現できるかを比べる（セクション 5）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、回答がどう変わるかを見る（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、用語集データの分割、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、評価範囲の確認、提出 | 学習時間はデータ件数によります |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データは用語集の質問・回答ペアです。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.2（2026-11-01）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、セクション2で作った train 分割の全件を20周する設定です。1エポックあたりのステップ数はデータ件数とバッチサイズで決まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # セクション2で分けた train データを何周学習するか
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 用語集データの読み込みと train / test 分割

**やること**: `glossary_sft.jsonl` を読み込み、学習用 `train_records`、評価用 `test`、正解 `gold` を作ります。

- assistant の定義文が同じレコードを同じ用語の質問表現としてまとめます。各定義につき1件目を学習、2件目をテストに使います。
- test の質問文は学習に含めませんが、定義文自体は学習データにもあります。この評価は未知の知識ではなく、学習した用語に対する質問の言い換えへの応答を測ります。
- `gold` は test の質問文から正解の定義文を引く辞書です。モデルへの入力には正解を含めません。
- Colab では `glossary_sft.jsonl` を `/content` にアップロードしてください。ローカル実行では作業ディレクトリまたはその親ディレクトリも検索します。

**見るところ**: train / test / gold の件数と、学習例・テスト例が1件ずつ表示されます。質問文が重複した場合や、各定義の質問が2件でない場合はエラーになります。

In [ ]:
sft_data_path = next(
    (path for path in (Path("glossary_sft.jsonl"), Path("../glossary_sft.jsonl"), Path("/content/glossary_sft.jsonl")) if path.is_file()),
    None,
)
if sft_data_path is None:
    raise FileNotFoundError("glossary_sft.jsonl をノートブックの作業ディレクトリか /content に置いてください")
with sft_data_path.open(encoding="utf-8") as f:
    sft_records = [json.loads(line) for line in f if line.strip()]

examples_by_answer = {}
for record in sft_records:
    messages = record.get("messages", [])
    if len(messages) != 2 or [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError("各JSONLレコードは user / assistant の2メッセージにしてください")
    if any(not isinstance(message.get("content"), str) for message in messages):
        raise ValueError("messages の content は文字列にしてください")
    examples_by_answer.setdefault(messages[1]["content"], []).append(record)

train_records, test, gold = [], [], {}
for answer, examples in examples_by_answer.items():
    if len(examples) != 2:
        raise ValueError(f"各定義の質問例は2件必要です: {answer[:40]}")
    train_records.append(examples[0])
    test.append(examples[1])
    question = examples[1]["messages"][0]["content"]
    gold[question] = answer

train_questions = {record["messages"][0]["content"] for record in train_records}
test_questions = [record["messages"][0]["content"] for record in test]
assert train_questions.isdisjoint(test_questions), "train と test で質問文が重複しています"
assert len(test_questions) == len(set(test_questions)), "test 内で質問文が重複しています"
print(f"train={len(train_records)} test={len(test)} gold={len(gold)} definitions={len(examples_by_answer)}")
print("学習例:", train_records[0]["messages"])
print("テスト例:", test[0]["messages"][0], "| 正解:", gold[test[0]["messages"][0]["content"]])
(WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train_records), encoding="utf-8")
(WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の質問表現に回答させます。入力には user メッセージだけを使い、assistant の正解文はモデルに渡しません。

**採点のしかた**（セクション 5 でも同じ関数を使います）: 生成文と `gold` の定義文を比較し、前後の空白や連続する空白を正規化したうえで完全一致かを判定します。言い換えた回答は不一致になるため、表の出力も読んでください。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: test の全質問と正解、学習前の出力、および完全一致の判定が表示されます。出力は `/content/qlora_lab/eval_before.json` にも保存されます。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: 公開モデルはログインなしでダウンロードできます。利用規約への同意が必要なモデルや非公開リポジトリにはログインが必要です。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層に高速な実装がないため、遅いが正しい実装で動くという案内です。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def generate(model, example):
    messages = example["messages"][:-1]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def normalize_answer(text: str) -> str:
    return " ".join(text.split())

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の質問に回答させ、正解の定義との完全一致率を集計する。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for example in test:
        question = example["messages"][0]["content"]
        raw = generate(model, example)
        expected = gold[question]
        exact_match = normalize_answer(raw) == normalize_answer(expected)
        rows.append({"question": question, "raw": raw, "gold": expected, "exact_match": exact_match})
    n = len(rows)
    summary = {"label": label, "n": n,
               "exact_match_rate": sum(row["exact_match"] for row in rows) / n,
               "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"質問": row["question"], "正解": row["gold"], "学習前の出力": row["raw"], "完全一致": mark(row["exact_match"])} for row in before_rows]))

## 4. LoRA 学習（所要時間はデータ件数によります）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上でadapterだけを学習します。量子化でメモリを節約できますが、基盤モデルの出力が変わる場合があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確認できます。

### 学習の実行

**やること**: セクション2で `glossary_sft.jsonl` を train / test に分けます。学習セルでは train 側の `messages` を加工せず `Dataset.from_list(train_records)` に渡してLoRA学習します。Colabではファイルを `/content` にアップロードしてから実行してください。学習したadapterは `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件です。1 エポックあたりのステップ数は train 件数で決まり、同じ例を何回学習するかは `num_train_epochs` で決まります。
- 学習後のモデルは変数 `tuned` に入ります。学習前の評価結果はセクション3で保存され、セクション5で学習後と比較します。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表: 学習中に損失が下がるかを確認します。データ件数によりステップ数と所要時間は変わります。
- `学習時間: ... 秒 | ステップ数: ... | エポック: ...`: 実際に学習した件数と周回数を確認します。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(train_records)
print(f"学習データ: {len(train_ds)} 件")

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから設定済みの学習ステップまで続きを学習します。必要なステップ数は train 件数とエポック数で決まります。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、セクション3と同じ test 質問・生成設定を使って評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `exact_match_rate`（空白を正規化した定義文の完全一致率）が学習前からどう変わったか。
- `退行`: 学習前は正解と一致したのに、学習後に一致しなくなった質問です。
- 質問ごとの表: 正解と学習前後の回答を読み、完全一致でない回答がどのように異なるか確認してください。

テストでは質問の言い回しだけを学習から外しています。正解の定義自体は学習データにも含まれるため、このスコアは未知の用語の知識を測るものではありません。

seed を固定しても、GPU 上の学習は完全には再現しません。スコアや残る質問が実行ごとに変わる場合があります。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[ ["n", "exact_match_rate", "seconds"] ]
display(compare)
# 学習前は一致したのに、学習後に一致しなくなった質問
regressed = [before["question"] for before, after in zip(before_rows, after_rows) if before["exact_match"] and not after["exact_match"]]
print("退行:", regressed)

# 質問・正解・学習前後の出力を並べ、言い換えに対する応答を確認する
detail = pd.DataFrame([{"質問": before["question"], "正解": before["gold"],
                        "学習前の出力": before["raw"], "学習前一致": mark(before["exact_match"]),
                        "学習後の出力": after["raw"], "学習後一致": mark(after["exact_match"])}
                       for before, after in zip(before_rows, after_rows)])
display(detail)

## 6. 評価範囲の確認

この test は、各用語について学習に使わなかった質問表現で回答できるかを測ります。各正解の定義文は train にも含まれているため、未知の用語に対する知識や、用語集にない質問への回答・保留の適切さは評価していません。

未知の知識やハルシネーションを評価したい場合は、用語集にない質問と「回答できない」という正解を別途用意し、train に混ぜずに test に加える必要があります。

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`、`test.jsonl`、`gold.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、完全一致しなかった回答の説明
- 評価が質問表現の言い換えを対象とし、未知の用語の知識を測っていないこと
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じモデルを 4bit（NF4）で読み込み、重みのメモリと、学習前の出力がどう変わるかを見ます。QLoRA は、この 4bit の基盤モデルに LoRA を付けて学習する方法です。ここでは学習はしません。

- bitsandbytes は CUDA の GPU が前提なので、GPU ランタイムでだけ動きます。
- 4bit の読み込みは `device_map="auto"` で GPU に配置します（bitsandbytes の読み込み方式に合わせています）。
- 4bit 側で学習まで試す場合は、`prepare_model_for_kbit_training(model_4bit)` を通してからセクション 4 と同じ手順で学習し、学習前後とも 4bit の基盤モデルで評価します。

**見るところ**

- 重みのメモリ: 32bit の約 3GB に対して 4bit は 1GB 未満になります。
- 比較表と質問ごとの表: 学習していないので、32bit の学習前（`before`）との差は量子化の影響です。`exact_match_rate` と出力原文を確認し、量子化で回答がどのように変わるかを比べてください。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "exact_match_rate", "seconds"]])
display(pd.DataFrame([{"質問": row["question"], "正解": row["gold"], "4bit・学習前の出力": row["raw"],
                       "完全一致": mark(row["exact_match"])} for row in q_rows]))